# 🍔 Nutritional Profiling of Fast Food Chains
### A Data-Driven Analysis Using Machine Learning Regression Models
**DSA-609 Data Science Module — Case Study Mini Project**

---

**Research Question:**
> *How do fast food nutritional profiles differ across restaurant chains, and can we accurately predict calorie content from other nutritional variables?*

---

### Dataset
- **Source:** [Tidy Tuesday Fast Food Calories](https://github.com/rfordatascience/tidytuesday)
- **Size:** 515 items from 8 restaurants
- **Restaurants:** McDonald's, Chick-fil-A, Arby's, Dairy Queen, Burger King, Subway, Taco Bell, Sonic
- **Variables:** 18 columns including calories, fats, carbs, proteins, vitamins, minerals

## 📦 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

# Set plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'DejaVu Sans'

print(f'TensorFlow version: {tf.__version__}')
print('All libraries imported successfully ✓')

---
## 📥 2. Data Acquisition

In [ ]:
# ─── Option A: Load from local file ───────────────────────────────────────────
# df = pd.read_csv('fastfood_calories.csv')

# ─── Option B: Load directly from Tidy Tuesday GitHub ─────────────────────────
url = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2018/2018-09-04/fastfood_calories.csv'
df = pd.read_csv(url)

print('Dataset loaded successfully ✓')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

In [ ]:
# Preview first 5 rows
df.head()

In [ ]:
# Basic dataset info
print('=== Dataset Info ===')
df.info()
print(f'\nShape: {df.shape[0]} rows × {df.shape[1]} columns')

---
## 🧹 3. Data Cleaning and Preprocessing

In [ ]:
# ─── Step 1: Remove unnecessary columns ───────────────────────────────────────
cols_to_drop = [col for col in ['Unnamed: 0', 'salad'] if col in df.columns]
df = df.drop(columns=cols_to_drop)
print(f'Step 1 ✓ | Dropped columns: {cols_to_drop}')
print(f'  Shape after drop: {df.shape}')

In [ ]:
# ─── Step 2: Standardise restaurant names ─────────────────────────────────────
df['restaurant'] = df['restaurant'].str.strip()
name_map = {
    'Mcdonalds':    'McDonalds',
    'Chick Fil-A':  'Chick-fil-A',
    'Arbys':        "Arby's",
    'Dairy Queen':  'Dairy Queen',
    'Burger King':  'Burger King',
    'Subway':       'Subway',
    'Taco Bell':    'Taco Bell',
    'Sonic':        'Sonic'
}
df['restaurant'] = df['restaurant'].replace(name_map)

print('Step 2 ✓ | Restaurant names standardised')
print(f'  Unique restaurants: {sorted(df["restaurant"].unique())}')

In [ ]:
# ─── Step 3: Clean item names ─────────────────────────────────────────────────
df['item'] = df['item'].str.strip()
print('Step 3 ✓ | Item names cleaned (whitespace stripped)')

In [ ]:
# ─── Step 4: Handle missing values ────────────────────────────────────────────
numeric_cols = [
    'calories', 'cal_fat', 'total_fat', 'sat_fat', 'trans_fat',
    'cholesterol', 'sodium', 'total_carb', 'fiber', 'sugar',
    'protein', 'vit_a', 'vit_c', 'calcium'
]

# Replace string 'NA' with actual NaN
df[numeric_cols] = df[numeric_cols].replace('NA', pd.NA)

# Check missing values before filling
print('Step 4 | Missing values before fill:')
missing = df[numeric_cols].isnull().sum()
print(missing[missing > 0].to_string())

# Create cleaned dataframe
cleaned_df = df.copy()
cleaned_df[numeric_cols] = cleaned_df[numeric_cols].fillna(0)

print('\nStep 4 ✓ | Missing values filled with 0')
print(f'  Remaining missing: {cleaned_df[numeric_cols].isnull().sum().sum()}')

In [ ]:
# ─── Step 5: Convert to numeric types ────────────────────────────────────────
cleaned_df[numeric_cols] = cleaned_df[numeric_cols].apply(pd.to_numeric, errors='coerce')
print('Step 5 ✓ | Columns converted to numeric types')

In [ ]:
# ─── Step 6: Remove duplicates ───────────────────────────────────────────────
print(f'Step 6 | Duplicate rows found: {cleaned_df.duplicated().sum()}')
cleaned_df.drop_duplicates(inplace=True)
cleaned_df.reset_index(drop=True, inplace=True)
print(f'Step 6 ✓ | Duplicates removed')
print(f'\n✅ Final cleaned dataset: {cleaned_df.shape[0]} rows × {cleaned_df.shape[1]} columns')

In [ ]:
# Summary statistics
print('=== Descriptive Statistics ===')
cleaned_df[numeric_cols].describe().round(2)

---
## 📊 4. Exploratory Data Analysis (EDA)

### 4.1 Calorie Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
sns.histplot(data=cleaned_df, x='calories', bins=30, kde=True, color='steelblue', ax=axes[0])
axes[0].set_title('Distribution of Calories in Fast Food Items', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Calories (kcal)')
axes[0].set_ylabel('Frequency')
axes[0].axvline(cleaned_df['calories'].mean(), color='red', linestyle='--', label=f'Mean: {cleaned_df["calories"].mean():.0f} kcal')
axes[0].axvline(cleaned_df['calories'].median(), color='orange', linestyle='--', label=f'Median: {cleaned_df["calories"].median():.0f} kcal')
axes[0].legend()

# Boxplot
sns.boxplot(data=cleaned_df, x='calories', color='lightsteelblue', ax=axes[1])
axes[1].set_title('Boxplot of Calories', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Calories (kcal)')

plt.tight_layout()
plt.savefig('fig1_calorie_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Mean calories:   {cleaned_df["calories"].mean():.1f} kcal')
print(f'Median calories: {cleaned_df["calories"].median():.1f} kcal')
print(f'Std deviation:   {cleaned_df["calories"].std():.1f} kcal')
print(f'Max calories:    {cleaned_df["calories"].max():.0f} kcal')
print(f'Min calories:    {cleaned_df["calories"].min():.0f} kcal')

### 4.2 Average Calories by Restaurant Chain

In [ ]:
avg_calories = cleaned_df.groupby('restaurant')['calories'].mean().sort_values()

colors = ['#2ecc71' if v < 450 else '#e67e22' if v < 600 else '#e74c3c' for v in avg_calories.values]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(avg_calories.index, avg_calories.values, color=colors, edgecolor='white', linewidth=0.8)

# Add value labels on bars
for bar, val in zip(bars, avg_calories.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{val:.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Average Calories per Restaurant Chain', fontsize=13, fontweight='bold')
ax.set_xlabel('Restaurant')
ax.set_ylabel('Average Calories (kcal)')
ax.set_ylim(0, avg_calories.max() * 1.15)

# Legend
legend_elements = [
    mpatches.Patch(facecolor='#2ecc71', label='Low  (<450 kcal)'),
    mpatches.Patch(facecolor='#e67e22', label='Mid  (450–600 kcal)'),
    mpatches.Patch(facecolor='#e74c3c', label='High (>600 kcal)')
]
ax.legend(handles=legend_elements, loc='upper left')

plt.tight_layout()
plt.savefig('fig2_avg_calories_by_chain.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Average Calories per Restaurant ===')
print(avg_calories.to_string())

### 4.3 Number of Items per Restaurant

In [ ]:
item_counts = cleaned_df['restaurant'].value_counts()

fig, ax = plt.subplots(figsize=(11, 5))
sns.barplot(x=item_counts.index, y=item_counts.values, palette='Blues_r', ax=ax)

for i, (idx, val) in enumerate(item_counts.items()):
    ax.text(i, val + 1.5, str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_title('Number of Menu Items per Restaurant', fontsize=13, fontweight='bold')
ax.set_xlabel('Restaurant')
ax.set_ylabel('Number of Items')
ax.set_ylim(0, item_counts.max() * 1.15)

plt.tight_layout()
plt.savefig('fig3_items_per_restaurant.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Item Counts ===')
print(item_counts.to_string())

### 4.4 Restaurant-wise Nutritional Comparison

In [ ]:
# Scale sodium to grams for comparable visualisation
data_plot = cleaned_df.copy()
data_plot['sodium_g'] = data_plot['sodium'] / 1000

restaurant_stats = data_plot.groupby('restaurant')[['protein', 'sodium_g', 'total_fat']].mean()
restaurant_stats.columns = ['Protein (g)', 'Sodium (g)', 'Total Fat (g)']

print('=== Restaurant-wise Nutrient Averages ===')
print(restaurant_stats.round(2).to_string())

ax = restaurant_stats.plot(kind='bar', figsize=(12, 6), width=0.7,
                            color=['#3498db', '#e74c3c', '#2ecc71'])
ax.set_title('Restaurant-wise Comparison of Key Nutrients', fontsize=13, fontweight='bold')
ax.set_ylabel('Amount (g)')
ax.set_xlabel('Restaurant')
plt.xticks(rotation=30, ha='right')
plt.legend(title='Nutrient')
plt.tight_layout()
plt.savefig('fig4_restaurant_nutrients.png', dpi=150, bbox_inches='tight')
plt.show()

### 4.5 Correlation Analysis

In [ ]:
corr_cols = ['calories', 'protein', 'sodium', 'total_fat', 'total_carb', 'cholesterol']
corr_matrix = cleaned_df[corr_cols].corr()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Heatmap
mask = np.zeros_like(corr_matrix, dtype=bool)
mask[np.triu_indices_from(mask, k=1)] = False
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            linewidths=0.5, ax=axes[0], vmin=-1, vmax=1,
            annot_kws={'size': 10})
axes[0].set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')

# Scatter: Calories vs Total Fat
scatter = axes[1].scatter(
    cleaned_df['total_fat'], cleaned_df['calories'],
    c=cleaned_df.groupby('restaurant').ngroup().loc[cleaned_df.index],
    cmap='tab10', alpha=0.6, s=30
)
axes[1].set_xlabel('Total Fat (g)', fontsize=11)
axes[1].set_ylabel('Calories (kcal)', fontsize=11)
axes[1].set_title(f'Calories vs Total Fat  (r = {corr_matrix.loc["calories", "total_fat"]:.2f})',
                   fontsize=13, fontweight='bold')

# Trend line
z = np.polyfit(cleaned_df['total_fat'], cleaned_df['calories'], 1)
p = np.poly1d(z)
x_line = np.linspace(cleaned_df['total_fat'].min(), cleaned_df['total_fat'].max(), 100)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=1.5, label='Trend line')
axes[1].legend()

plt.tight_layout()
plt.savefig('fig5_correlation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Key Correlations with Calories ===')
print(corr_matrix['calories'].drop('calories').sort_values(ascending=False).round(3).to_string())

### 4.6 Scatter Plots: Calories vs Key Nutrients

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
nutrients = [('total_fat', 'Total Fat (g)'), ('sodium', 'Sodium (mg)'), ('protein', 'Protein (g)')]

restaurants = cleaned_df['restaurant'].unique()
palette = sns.color_palette('tab10', len(restaurants))
color_map = dict(zip(restaurants, palette))

for ax, (col, label) in zip(axes, nutrients):
    for rest in restaurants:
        subset = cleaned_df[cleaned_df['restaurant'] == rest]
        ax.scatter(subset[col], subset['calories'],
                   label=rest, color=color_map[rest], alpha=0.6, s=25)
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Calories (kcal)', fontsize=11)
    r = cleaned_df[[col, 'calories']].corr().iloc[0, 1]
    ax.set_title(f'Calories vs {label}\n(r = {r:.2f})', fontsize=11, fontweight='bold')

# Single shared legend
handles = [mpatches.Patch(color=color_map[r], label=r) for r in restaurants]
fig.legend(handles=handles, loc='lower center', ncol=4, bbox_to_anchor=(0.5, -0.12),
           title='Restaurant', fontsize=9)

plt.tight_layout()
plt.savefig('fig6_scatter_plots.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🔬 5. Principal Component Analysis (PCA)

In [ ]:
# Select nutritional features for PCA
pca_cols = ['total_fat', 'cholesterol', 'sodium', 'total_carb', 'protein', 'vit_a', 'vit_c', 'calcium']
nutrition_df = cleaned_df[pca_cols]

# Standardise
scaler = StandardScaler()
nutrition_scaled = scaler.fit_transform(nutrition_df)

# Fit PCA
pca = PCA()
principal_components = pca.fit_transform(nutrition_scaled)

eigenvalues = pca.explained_variance_
proportions = pca.explained_variance_ratio_
cumulative  = np.cumsum(proportions)

print('=== PCA Results ===')
print(f'{"PC":<6} {"Eigenvalue":<14} {"Variance %":<14} {"Cumulative %"}')
print('-' * 50)
for i, (ev, prop, cum) in enumerate(zip(eigenvalues, proportions, cumulative)):
    print(f'PC{i+1:<5} {ev:<14.4f} {prop*100:<14.2f} {cum*100:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
axes[0].plot(range(1, len(eigenvalues)+1), eigenvalues, 'bo-', linewidth=2, markersize=7)
axes[0].axhline(y=1, color='r', linestyle='--', label='Kaiser criterion (λ=1)')
axes[0].set_title('Scree Plot', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Eigenvalue (Explained Variance)')
axes[0].legend()
axes[0].grid(True, alpha=0.4)

# Cumulative variance
bar_colors = ['#2ecc71' if c <= 0.8 else '#e67e22' for c in cumulative]
axes[1].bar(range(1, len(proportions)+1), proportions*100, color='steelblue',
            alpha=0.7, label='Individual')
ax2 = axes[1].twinx()
ax2.plot(range(1, len(cumulative)+1), cumulative*100, 'ro-', linewidth=2, label='Cumulative')
ax2.axhline(y=80, color='gray', linestyle='--', alpha=0.5, label='80% threshold')
ax2.set_ylabel('Cumulative Variance (%)')
axes[1].set_title('Explained Variance per Component', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Principal Component')
axes[1].set_ylabel('Individual Variance (%)')
axes[1].legend(loc='upper left')
ax2.legend(loc='center right')

plt.tight_layout()
plt.savefig('fig7_pca_scree.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PCA Loadings (Eigenvectors)
loadings = pd.DataFrame(
    pca.components_[:3].T,
    index=pca_cols,
    columns=['PC1', 'PC2', 'PC3']
).round(4)

print('=== PCA Loadings (First 3 Components) ===')
print(loadings.to_string())

# Heatmap of loadings
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(loadings, annot=True, fmt='.3f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, vmin=-0.7, vmax=0.7)
ax.set_title('PCA Feature Loadings (PC1–PC3)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('fig8_pca_loadings.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# PCA Biplot (PC1 vs PC2)
pc_df = pd.DataFrame(principal_components[:, :2], columns=['PC1', 'PC2'])
pc_df['restaurant'] = cleaned_df['restaurant'].values

fig, ax = plt.subplots(figsize=(10, 7))
for rest in restaurants:
    subset = pc_df[pc_df['restaurant'] == rest]
    ax.scatter(subset['PC1'], subset['PC2'], label=rest,
               color=color_map[rest], alpha=0.5, s=30)

# Loading arrows (scaled)
scale = 3.5
for i, feature in enumerate(pca_cols):
    ax.annotate('', xy=(pca.components_[0, i]*scale, pca.components_[1, i]*scale),
                xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
    ax.text(pca.components_[0, i]*scale*1.15, pca.components_[1, i]*scale*1.15,
            feature, color='red', fontsize=8, ha='center')

ax.set_xlabel(f'PC1 ({proportions[0]*100:.1f}% variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({proportions[1]*100:.1f}% variance)', fontsize=11)
ax.set_title('PCA Biplot: PC1 vs PC2 by Restaurant', fontsize=13, fontweight='bold')
ax.axhline(0, color='gray', linewidth=0.5)
ax.axvline(0, color='gray', linewidth=0.5)
ax.legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.savefig('fig9_pca_biplot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 🤖 6. Machine Learning: Calorie Prediction Models

In [ ]:
# ─── Feature and Target Selection ─────────────────────────────────────────────
feature_cols = ['total_fat', 'cholesterol', 'sodium', 'total_carb', 'protein', 'vit_a', 'vit_c', 'calcium']
target_col   = 'calories'

X = cleaned_df[feature_cols]
y = cleaned_df[target_col]

# ─── Train / Test Split (70 / 30) ─────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=88
)

print(f'Training samples: {X_train.shape[0]}')
print(f'Testing  samples: {X_test.shape[0]}')
print(f'Features used: {feature_cols}')

### 6.1 Linear Regression

In [ ]:
# Train
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predict
lr_pred = lr_model.predict(X_test)

# Evaluate
lr_mae  = mean_absolute_error(y_test, lr_pred)
lr_mse  = mean_squared_error(y_test, lr_pred)
lr_r2   = r2_score(y_test, lr_pred)

print('=== Linear Regression Performance ===')
print(f'  R²  Score : {lr_r2:.4f}')
print(f'  MAE (kcal): {lr_mae:.2f}')
print(f'  MSE (kcal²): {lr_mse:.2f}')

print('\n=== Feature Coefficients ===')
coef_df = pd.DataFrame({'Feature': feature_cols, 'Coefficient': lr_model.coef_})
coef_df = coef_df.reindex(coef_df['Coefficient'].abs().sort_values(ascending=False).index)
print(coef_df.to_string(index=False))

In [ ]:
# Actual vs Predicted plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: actual vs predicted
axes[0].scatter(y_test, lr_pred, alpha=0.5, color='steelblue', s=25)
min_val, max_val = min(y_test.min(), lr_pred.min()), max(y_test.max(), lr_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5, label='Perfect prediction')
axes[0].set_xlabel('Actual Calories (kcal)')
axes[0].set_ylabel('Predicted Calories (kcal)')
axes[0].set_title(f'Linear Regression — Actual vs Predicted\n(R² = {lr_r2:.4f})', fontweight='bold')
axes[0].legend()

# Residuals
residuals = y_test.values - lr_pred
axes[1].scatter(lr_pred, residuals, alpha=0.5, color='coral', s=25)
axes[1].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Predicted Calories (kcal)')
axes[1].set_ylabel('Residuals (kcal)')
axes[1].set_title('Linear Regression — Residual Plot', fontweight='bold')

plt.tight_layout()
plt.savefig('fig10_lr_results.png', dpi=150, bbox_inches='tight')
plt.show()

### 6.2 Random Forest Regressor

In [ ]:
# Train
rf_model = RandomForestRegressor(n_estimators=100, random_state=88, n_jobs=-1)
rf_model.fit(X_train, y_train)

# Predict
rf_pred = rf_model.predict(X_test)

# Evaluate
rf_mae  = mean_absolute_error(y_test, rf_pred)
rf_mse  = mean_squared_error(y_test, rf_pred)
rf_r2   = r2_score(y_test, rf_pred)

print('=== Random Forest Regressor Performance ===')
print(f'  R²  Score : {rf_r2:.4f}')
print(f'  MAE (kcal): {rf_mae:.2f}')
print(f'  MSE (kcal²): {rf_mse:.2f}')

In [ ]:
# Feature importance
importance_df = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Feature importance
axes[0].barh(importance_df['Feature'], importance_df['Importance'],
             color='steelblue', edgecolor='white')
axes[0].set_title('Random Forest — Feature Importances', fontweight='bold')
axes[0].set_xlabel('Importance Score')

# Actual vs Predicted
axes[1].scatter(y_test, rf_pred, alpha=0.5, color='green', s=25)
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5)
axes[1].set_xlabel('Actual Calories (kcal)')
axes[1].set_ylabel('Predicted Calories (kcal)')
axes[1].set_title(f'Random Forest — Actual vs Predicted\n(R² = {rf_r2:.4f})', fontweight='bold')

plt.tight_layout()
plt.savefig('fig11_rf_results.png', dpi=150, bbox_inches='tight')
plt.show()

print('=== Feature Importances ===')
print(importance_df.sort_values('Importance', ascending=False).to_string(index=False))

### 6.3 Neural Network Model

In [ ]:
# Scale features for neural network
feature_scaler = StandardScaler()
X_train_scaled = feature_scaler.fit_transform(X_train)
X_test_scaled  = feature_scaler.transform(X_test)

# Build model
tf.random.set_seed(88)

nn_model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer (linear for regression)
])

nn_model.compile(
    optimizer='adam',
    loss='mse',
    metrics=['mae']
)

nn_model.summary()

In [ ]:
# Early stopping to prevent overfitting
early_stop = EarlyStopping(
    monitor='val_loss', patience=15, restore_best_weights=True, verbose=0
)

# Train
history = nn_model.fit(
    X_train_scaled, y_train,
    epochs=100,
    batch_size=16,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop],
    verbose=1
)

print(f'\nTraining stopped at epoch {len(history.history["loss"])}')

In [ ]:
# Evaluate
nn_pred = nn_model.predict(X_test_scaled).flatten()

nn_mae  = mean_absolute_error(y_test, nn_pred)
nn_mse  = mean_squared_error(y_test, nn_pred)
nn_r2   = r2_score(y_test, nn_pred)

print('=== Neural Network Performance ===')
print(f'  R²  Score : {nn_r2:.4f}')
print(f'  MAE (kcal): {nn_mae:.2f}')
print(f'  MSE (kcal²): {nn_mse:.2f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training history
axes[0].plot(history.history['loss'], label='Train Loss', color='steelblue')
axes[0].plot(history.history['val_loss'], label='Val Loss', color='coral')
axes[0].set_title('Neural Network — Training History', fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Actual vs Predicted
axes[1].scatter(y_test, nn_pred, alpha=0.5, color='purple', s=25)
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=1.5)
axes[1].set_xlabel('Actual Calories (kcal)')
axes[1].set_ylabel('Predicted Calories (kcal)')
axes[1].set_title(f'Neural Network — Actual vs Predicted\n(R² = {nn_r2:.4f})', fontweight='bold')

plt.tight_layout()
plt.savefig('fig12_nn_results.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 📋 7. Model Comparison and Summary

In [ ]:
# Comparison table
results = pd.DataFrame({
    'Model':            ['Linear Regression', 'Random Forest', 'Neural Network'],
    'R² Score':         [round(lr_r2, 4), round(rf_r2, 4), round(nn_r2, 4)],
    'MAE (kcal)':       [round(lr_mae, 2), round(rf_mae, 2), round(nn_mae, 2)],
    'MSE (kcal²)':      [round(lr_mse, 2), round(rf_mse, 2), round(nn_mse, 2)],
    'Best Performer':   ['✓', '', '']
})

# Highlight best per metric
best_r2  = results['R² Score'].idxmax()
best_mae = results['MAE (kcal)'].idxmin()
best_mse = results['MSE (kcal²)'].idxmin()
results.loc[best_r2,  'Best R²']   = '✓'
results.loc[best_mae, 'Best MAE']  = '✓'
results.loc[best_mse, 'Best MSE']  = '✓'

print('=' * 70)
print('MODEL PERFORMANCE COMPARISON')
print('=' * 70)
print(results[['Model', 'R² Score', 'MAE (kcal)', 'MSE (kcal²)']].to_string(index=False))
print('=' * 70)
print(f'Best R²  → {results.loc[best_r2,  "Model"]}')
print(f'Best MAE → {results.loc[best_mae, "Model"]}')
print(f'Best MSE → {results.loc[best_mse, "Model"]}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
models = ['Linear\nRegression', 'Random\nForest', 'Neural\nNetwork']
palette = ['#2ecc71', '#3498db', '#9b59b6']

# R² comparison
bars0 = axes[0].bar(models, [lr_r2, rf_r2, nn_r2], color=palette, edgecolor='white')
for bar, val in zip(bars0, [lr_r2, rf_r2, nn_r2]):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.03,
                 f'{val:.4f}', ha='center', va='top', fontsize=10, fontweight='bold', color='white')
axes[0].set_title('R² Score (Higher is Better)', fontweight='bold')
axes[0].set_ylim(0.9, 1.0)
axes[0].set_ylabel('R²')

# MAE comparison
bars1 = axes[1].bar(models, [lr_mae, rf_mae, nn_mae], color=palette, edgecolor='white')
for bar, val in zip(bars1, [lr_mae, rf_mae, nn_mae]):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_title('MAE (Lower is Better)', fontweight='bold')
axes[1].set_ylabel('MAE (kcal)')

# MSE comparison
bars2 = axes[2].bar(models, [lr_mse, rf_mse, nn_mse], color=palette, edgecolor='white')
for bar, val in zip(bars2, [lr_mse, rf_mse, nn_mse]):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 20,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[2].set_title('MSE (Lower is Better)', fontweight='bold')
axes[2].set_ylabel('MSE (kcal²)')

plt.suptitle('Model Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('fig13_model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 💡 8. Key Findings and Conclusions

### Research Question Answered:

**Q: How do fast food nutritional profiles differ across restaurant chains?**

- **McDonald's** has the highest average calories (640 kcal) and leads in protein (40.3g) and sodium (1,438mg), reflecting a calorie-dense, protein-forward menu strategy.
- **Chick-fil-A** offers the lowest average calories (384 kcal) with lean protein options — the healthiest overall profile.
- **Burger King** and **Sonic** have the highest total fat content (36.8g and 37.6g respectively).
- **PCA** revealed that nutritional variation is driven by two principal axes: a macronutrient/caloric axis (PC1, 45.25%) and a micronutrient/vitamin axis (PC2, 21.59%).

**Q: Can we accurately predict calorie content from other nutritional variables?**

- **Yes** — Linear Regression achieved R² = 0.992, meaning 99.2% of calorie variance is explained by the nutritional features.
- The MAE of ~11.73 kcal represents a ~2.2% error relative to the mean calorie value (531 kcal).
- **Total fat** is the single strongest calorie predictor (r = 0.90), consistent with its 9 kcal/g energy density vs 4 kcal/g for protein and carbohydrates.
- Linear Regression outperformed both Random Forest (R²=0.96) and Neural Network (R²=0.98), demonstrating that model complexity is not always beneficial on fundamentally linear datasets.

---
## 📚 References

- Chollet, F. (2018). *Deep Learning with Python*. Manning Publications.
- Géron, A. (2023). *Hands-On Machine Learning with Scikit-Learn, Keras, and TensorFlow* (3rd ed.). O'Reilly Media.
- IBM. (2023a). *Data Analysis with Python*. Coursera. https://coursera.org/verify/RYUZRMMCAC6Z
- IBM. (2023b). *Machine Learning with Python*. Coursera. https://coursera.org/verify/3JRM9Z4F6C24
- McKinney, W. (2022). *Python for Data Analysis* (3rd ed.). O'Reilly Media.
- VanderPlas, J. (2017). *Python Data Science Handbook*. O'Reilly Media.
- Tidy Tuesday Dataset: https://github.com/rfordatascience/tidytuesday/tree/master/data/2018/2018-09-04